# 1. Создание и визуализация детектора

Сферический детектор радиуса `R` содержит `N` фотоумножителей. Центры расположены равномерно по сфере, а фотокатоды изображаются кругами радиуса `r` в касательных плоскостях.

In [7]:
%pip install -U numpy k3d ipywidgets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from math import cos, sin, acos, pi, sqrt
import numpy as np
import k3d

In [9]:
def detectors(n=40):
    points = []
    for i in range(n):
        phi = acos(1 - 2 * (i + 0.5) / n)
        theta = pi * (1 + sqrt(5)) * i
        points.append((cos(theta) * sin(phi), sin(theta) * sin(phi), cos(phi)))
    return points

In [10]:
class Detector:
    def __init__(self):
        self.R = 200.0
        self.N = 40
        self.r = 10.0
        self.a = 0.5
        self.pmts = detectors(self.N)

    def get_pmt_positions(self):
        return [(self.R * x, self.R * y, self.R * z) for x, y, z in self.pmts]

detector = Detector()
pmt_positions = np.array(detector.get_pmt_positions(), dtype=np.float32)

In [11]:
theta = np.linspace(0, np.pi, 100)
phi = np.linspace(0, 2 * np.pi, 100)
x = detector.R * np.outer(np.sin(theta), np.cos(phi))
y = detector.R * np.outer(np.sin(theta), np.sin(phi))
z = detector.R * np.outer(np.cos(theta), np.ones_like(phi))
vertices = np.column_stack((x.ravel(), y.ravel(), z.ravel())).astype(np.float32)
grid_indices = np.arange(x.size).reshape(x.shape)
faces = np.concatenate([
    np.stack((grid_indices[:-1, :-1], grid_indices[1:, :-1], grid_indices[1:, 1:]), axis=-1).reshape(-1, 3),
    np.stack((grid_indices[:-1, :-1], grid_indices[1:, 1:], grid_indices[:-1, 1:]), axis=-1).reshape(-1, 3)
]).astype(np.uint32)

normals = pmt_positions / np.linalg.norm(pmt_positions, axis=1, keepdims=True)
reference = np.tile(np.array([0, 0, 1], dtype=np.float32), (len(normals), 1))
reference[np.abs(normals[:, 2]) > 0.9] = [0, 1, 0]
basis_u = np.cross(normals, reference)
basis_u /= np.linalg.norm(basis_u, axis=1, keepdims=True)
basis_v = np.cross(normals, basis_u)
angles = np.linspace(0, 2 * np.pi, 33, endpoint=False, dtype=np.float32)
rim = pmt_positions[:, None, :] + detector.r * (
    np.cos(angles)[None, :, None] * basis_u[:, None, :]
    + np.sin(angles)[None, :, None] * basis_v[:, None, :]
)
disk_vertices = np.concatenate((pmt_positions[:, None, :], rim), axis=1).reshape(-1, 3).astype(np.float32)
center_indices = np.arange(detector.N, dtype=np.uint32) * (len(angles) + 1)
rim_indices = center_indices[:, None] + np.arange(1, len(angles) + 1, dtype=np.uint32)
disk_faces = np.stack((
    np.broadcast_to(center_indices[:, None], rim_indices.shape),
    rim_indices,
    np.roll(rim_indices, -1, axis=1)
), axis=-1).reshape(-1, 3)

plot = k3d.plot(grid_visible=False, camera_auto_fit=True)
plot += k3d.mesh(vertices, faces, color=0x4477AA, opacity=0.25)
plot += k3d.mesh(disk_vertices, disk_faces, color=0xFFD166, opacity=0.9)
plot += k3d.points(pmt_positions, point_size=0.15, color=0xFF0000)
plot.display()

Output()